# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaNehaBatool12/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose Random Forest for this modeling lane because the content-refresh dataset contains several different signals that may interact in nonlinear ways. The goal is to identify declining content that may need refreshing. Random Forest can combine multiple features without assuming a simple linear relationship and provides feature importance for interpretation. I will compare its performance against my Week-4 baseline using the same held-out data and metric. The result is intended as decision-support rather than a claim about Google's ranking algorithm.

## 2. Split design

I use a grouped train/test split based on client_id. Pages from the same client may share similar characteristics, so allowing one client's pages to appear in both training and test sets could make the evaluation overly optimistic. Holding out complete clients gives a more honest estimate of how the model may perform on unseen clients. I also exclude trend_direction and trend_pct from the model features because they directly describe the outcome and would cause target leakage.

In [8]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the same anonymized dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Target: whether the content is declining
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Grouped split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Total rows:", len(df))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print()
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

# Verify there is no client overlap
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap:", len(overlap))

print()
print("Training declining rate:",
      round(train_df["is_declining_label"].mean(), 3))
print("Test declining rate:",
      round(test_df["is_declining_label"].mean(), 3))


Total rows: 30000
Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7
Client overlap: 0

Training declining rate: 0.55
Test declining rate: 0.511


## 3. Train + compare vs my baseline



In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Features available at decision time
features = [
    "search_volume",
    "competition",
    "cpc",
    "engagement_rate",
    "scroll_rate"
]

# Keep only features that actually exist
features = [c for c in features if c in df.columns]

print("Features used:", features)

# Prepare train/test data from our grouped split
X_train = train_df[features].fillna(0)
X_test = test_df[features].fillna(0)

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Model predictions
model_pred = model.predict(X_test)

# Simple baseline:
# predict the most common training class for every test row
majority_class = int(y_train.mode()[0])
baseline_pred = [majority_class] * len(y_test)

# Compare using SAME test split and SAME metrics
results = pd.DataFrame({
    "Model": ["Week-4 Baseline", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, model_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})

print("\nModel vs Baseline")
print(results.round(3))


Features used: ['search_volume', 'competition', 'cpc', 'engagement_rate', 'scroll_rate']

Model vs Baseline
             Model  Accuracy  Precision  Recall     F1
0  Week-4 Baseline     0.511      0.511   1.000  0.676
1    Random Forest     0.526      0.541   0.475  0.506


## 4. Errors and interpretation
The Random Forest achieved slightly better accuracy than the baseline (0.526 vs 0.511), but its recall and F1 score were lower. The model made 2,921 incorrect predictions, including 1,269 false positives and 1,652 false negatives. This shows that the model misses some declining content and should be treated as decision-support rather than a perfect classifier.

Feature importance shows that the model relies most strongly on scroll_rate (0.557), followed by engagement_rate (0.176), search_volume (0.117), competition (0.078), and cpc (0.071). The results are directional: scroll and engagement behaviour appear to provide the strongest signal for identifying declining content.

In [10]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Confusion matrix
cm = confusion_matrix(y_test, model_pred)

tn, fp, fn, tp = cm.ravel()

print("Random Forest Error Analysis")
print("----------------------------")
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

print("\nFeature importance:")
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.round(3))

print("\nTotal prediction errors:",
      (model_pred != y_test).sum())

print("Error rate:",
      round((model_pred != y_test).mean(), 3))


Random Forest Error Analysis
----------------------------
True negatives : 1745
False positives: 1269
False negatives: 1652
True positives : 1497

Feature importance:
           feature  importance
4      scroll_rate       0.557
3  engagement_rate       0.176
0    search_volume       0.117
1      competition       0.078
2              cpc       0.071

Total prediction errors: 2921
Error rate: 0.474


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.